# 04 Prediction Analysis

This notebook analyzes prediction-vs-actual outputs with one goal in mind:

- show where feature engineering improves over baseline
- show which target/model-family combinations have monotonic improvement from baseline to `V0` to `V1` to `V2`
- show residual behavior by version and model family
- show where the models work well versus where they fail

## Imports And Load

In [15]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
RESULTS_DIR = ROOT / 'outputs' / 'results'
FIGURES_DIR = ROOT / 'outputs' / 'prediction_analysis_figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

results_df = pd.read_csv(RESULTS_DIR / 'model_results.csv')
eval_predictions = pd.read_parquet(RESULTS_DIR / 'eval_predictions_long.parquet')
best_eval_predictions = pd.read_parquet(RESULTS_DIR / 'best_eval_predictions.parquet')
results_df.head()

,run_id,target,model_version,model_family,feature_group,train_seasons,eval_season,mae,rmse,runtime_seconds,notes
0,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,season_avg_ast,2023-24,2024-25,1.372025,1.863006,0.013806,NaN
1,target_ast_BaselineA_RandomForestRegressor,target_ast,BaselineA,RandomForestRegressor,season_avg_ast,2023-24,2024-25,1.458695,1.998495,0.675006,NaN
2,target_ast_BaselineA_XGBRegressor,target_ast,BaselineA,XGBRegressor,season_avg_ast,2023-24,2024-25,1.378558,1.868841,0.319800,NaN
3,target_ast_BaselineB_LinearRegression,target_ast,BaselineB,LinearRegression,l5_avg_ast,2023-24,2024-25,1.406466,1.907371,0.009473,NaN
4,target_ast_BaselineB_RandomForestRegressor,target_ast,BaselineB,RandomForestRegressor,l5_avg_ast,2023-24,2024-25,1.409654,1.910595,0.368701,NaN


## Organize Versions Around A Strong Baseline

For each target and model family, choose the better of `BaselineA` and `BaselineB` as the baseline reference. Then compare:

- `baseline_ref`
- `V0`
- `V1`
- `V2`

In [16]:
def build_progression_table(results_df):
    rows = []
    for (target, model_family), g in results_df.groupby(['target', 'model_family']):
        g = g.copy()
        baseline_row = g[g['model_version'].isin(['BaselineA', 'BaselineB'])].sort_values('mae').iloc[0]
        baseline_ref = baseline_row['model_version']
        metrics = {row['model_version']: row['mae'] for _, row in g.iterrows()}
        row = {
            'target': target,
            'model_family': model_family,
            'baseline_ref': baseline_ref,
            'baseline_mae': baseline_row['mae'],
            'v0_mae': metrics['V0'],
            'v1_mae': metrics['V1'],
            'v2_mae': metrics['V2'],
            'monotonic_improvement': metrics['V2'] < metrics['V1'] < metrics['V0'] < baseline_row['mae'],
            'baseline_to_v2_mae_gain': baseline_row['mae'] - metrics['V2'],
            'baseline_to_v2_pct_gain': (baseline_row['mae'] - metrics['V2']) / baseline_row['mae'],
        }
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['target', 'model_family']).reset_index(drop=True)

progression_df = build_progression_table(results_df)
progression_df

,target,model_family,baseline_ref,baseline_mae,v0_mae,v1_mae,v2_mae,monotonic_improvement,baseline_to_v2_mae_gain,baseline_to_v2_pct_gain
0,target_ast,LinearRegression,BaselineA,1.372025,1.349590,1.352002,1.348317,False,0.023708,0.017279
1,target_ast,RandomForestRegressor,BaselineB,1.409654,1.390801,1.384068,1.378384,True,0.031270,0.022183
2,target_ast,XGBRegressor,BaselineA,1.378558,1.364798,1.357088,1.353155,True,0.025403,0.018427
3,target_pts,LinearRegression,BaselineA,4.748928,4.654190,4.640032,4.613657,True,0.135270,0.028484
4,target_pts,RandomForestRegressor,BaselineB,4.868520,4.767688,4.715854,4.670536,True,0.197983,0.040666
5,target_pts,XGBRegressor,BaselineA,4.759626,4.689317,4.679237,4.610939,True,0.148687,0.031239
6,target_reb,LinearRegression,BaselineA,1.969226,1.942623,1.943205,1.945609,False,0.023617,0.011993
7,target_reb,RandomForestRegressor,BaselineB,2.009972,1.982997,1.966291,1.965743,True,0.044229,0.022005
8,target_reb,XGBRegressor,BaselineA,1.974493,1.958050,1.945920,1.950483,False,0.024010,0.012160


## Monotonic Improvement Permutations

These are the combinations where the sequence improves cleanly:

- baseline
- `V0`
- `V1`
- `V2`

In [17]:
monotonic_df = progression_df[progression_df['monotonic_improvement']].copy()
monotonic_df[['target', 'model_family', 'baseline_ref', 'baseline_mae', 'v0_mae', 'v1_mae', 'v2_mae', 'baseline_to_v2_mae_gain', 'baseline_to_v2_pct_gain']]

,target,model_family,baseline_ref,baseline_mae,v0_mae,v1_mae,v2_mae,baseline_to_v2_mae_gain,baseline_to_v2_pct_gain
1,target_ast,RandomForestRegressor,BaselineB,1.409654,1.390801,1.384068,1.378384,0.031270,0.022183
2,target_ast,XGBRegressor,BaselineA,1.378558,1.364798,1.357088,1.353155,0.025403,0.018427
3,target_pts,LinearRegression,BaselineA,4.748928,4.654190,4.640032,4.613657,0.135270,0.028484
4,target_pts,RandomForestRegressor,BaselineB,4.868520,4.767688,4.715854,4.670536,0.197983,0.040666
5,target_pts,XGBRegressor,BaselineA,4.759626,4.689317,4.679237,4.610939,0.148687,0.031239
7,target_reb,RandomForestRegressor,BaselineB,2.009972,1.982997,1.966291,1.965743,0.044229,0.022005


## Build A Version-Normalized Evaluation Dataset

This creates a consistent long-format dataset with one `baseline_ref` version per target/model family so residual and error plots compare the same four stages.

In [18]:
frames = []
for row in progression_df.itertuples(index=False):
    subset = eval_predictions[(eval_predictions['target'] == row.target) & (eval_predictions['model_family'] == row.model_family)].copy()
    subset = subset[subset['model_version'].isin([row.baseline_ref, 'V0', 'V1', 'V2'])].copy()
    subset['version_stage'] = subset['model_version'].replace({row.baseline_ref: 'baseline_ref'})
    subset['baseline_ref_name'] = row.baseline_ref
    frames.append(subset)

analysis_eval = pd.concat(frames, ignore_index=True)
stage_order = ['baseline_ref', 'V0', 'V1', 'V2']
analysis_eval['version_stage'] = pd.Categorical(analysis_eval['version_stage'], categories=stage_order, ordered=True)
analysis_eval.head()

,player_id,player_name,game_id,game_date,season,team_abbr,opponent_team,home_away,actual,prediction,error,abs_error,squared_error,run_id,target,model_version,model_family,version_stage,baseline_ref_name
0,2544,LeBron James,0022400062,2024-10-22,2024-25,LAL,MIN,home,4,1.843977,-2.156023,2.156023,4.648434,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,baseline_ref,BaselineA
1,2544,LeBron James,0022400085,2024-10-25,2024-25,LAL,PHX,home,8,4.062517,-3.937483,3.937483,15.503770,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,baseline_ref,BaselineA
2,2544,LeBron James,0022400096,2024-10-26,2024-25,LAL,SAC,home,10,5.983727,-4.016273,4.016273,16.130447,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,baseline_ref,BaselineA
3,2544,LeBron James,0022400111,2024-10-28,2024-25,LAL,PHX,away,8,7.264534,-0.735466,0.735466,0.540911,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,baseline_ref,BaselineA
4,2544,LeBron James,0022400118,2024-10-30,2024-25,LAL,CLE,away,3,7.424635,4.424635,4.424635,19.577391,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,baseline_ref,BaselineA


## Error Progression Table

This table makes the feature-engineering story explicit by showing how MAE and RMSE change as feature sets become richer.

In [19]:
stage_summary = analysis_eval.groupby(['target', 'model_family', 'version_stage']).agg(
    n=('game_id', 'size'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: (s.mean()) ** 0.5),
    median_abs_error=('abs_error', 'median'),
    p90_abs_error=('abs_error', lambda s: s.quantile(0.9)),
).reset_index()
stage_summary.head(20)

,target,model_family,version_stage,n,mae,rmse,median_abs_error,p90_abs_error
0,target_ast,LinearRegression,baseline_ref,26306,1.372025,1.863006,1.036159,2.937483
1,target_ast,LinearRegression,V0,26306,1.349590,1.838608,1.011906,2.876542
2,target_ast,LinearRegression,V1,26306,1.352002,1.830642,1.033497,2.864879
3,target_ast,LinearRegression,V2,26306,1.348317,1.826779,1.026891,2.882087
4,target_ast,RandomForestRegressor,baseline_ref,26306,1.409654,1.910595,1.072534,3.058304
5,target_ast,RandomForestRegressor,V0,26306,1.390801,1.879293,1.056830,2.960596
6,target_ast,RandomForestRegressor,V1,26306,1.384068,1.862216,1.056539,2.952112
7,target_ast,RandomForestRegressor,V2,26306,1.378384,1.849359,1.051774,2.933125
8,target_ast,XGBRegressor,baseline_ref,26306,1.378558,1.868841,1.029394,2.961119
9,target_ast,XGBRegressor,V0,26306,1.364798,1.861474,1.015166,2.954318


## Improvement Plots

These plots are designed to show that the engineered feature sets improve over the baseline reference where the data supports that claim.

In [20]:
for target in sorted(stage_summary['target'].unique()):
    plot_df = stage_summary[stage_summary['target'] == target].copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(data=plot_df, x='version_stage', y='mae', hue='model_family', marker='o', ax=ax)
    ax.set_title(f'MAE Progression By Feature Version: {target}')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'mae_progression_{target}.png', dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(data=plot_df, x='version_stage', y='rmse', hue='model_family', marker='o', ax=ax)
    ax.set_title(f'RMSE Progression By Feature Version: {target}')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'rmse_progression_{target}.png', dpi=180)
    plt.close(fig)

sorted(p.name for p in FIGURES_DIR.glob('*progression*.png'))[:10]

['mae_progression_target_ast.png',
 'mae_progression_target_pts.png',
 'mae_progression_target_reb.png',
 'monotonic_mae_progression_target_ast.png',
 'monotonic_mae_progression_target_pts.png',
 'monotonic_mae_progression_target_reb.png',
 'rmse_progression_target_ast.png',
 'rmse_progression_target_pts.png',
 'rmse_progression_target_reb.png']

## Residual Graphs Across Versions X Model Families

For each target, create residual distributions grouped by feature version and separated by model family.

In [21]:
for target in sorted(analysis_eval['target'].unique()):
    plot_df = analysis_eval[analysis_eval['target'] == target].copy()
    g = sns.FacetGrid(plot_df, col='model_family', height=4, aspect=1.1, sharex=True, sharey=True)
    g.map_dataframe(sns.boxplot, x='version_stage', y='error', order=stage_order)
    g.set_titles('{col_name}')
    g.fig.subplots_adjust(top=0.8)
    g.fig.suptitle(f'Residual Distribution By Version And Model Family: {target}')
    g.savefig(FIGURES_DIR / f'residual_boxplot_{target}.png', dpi=180)
    plt.close(g.fig)

    g = sns.FacetGrid(plot_df, col='model_family', hue='version_stage', height=4, aspect=1.1, sharex=True, sharey=True)
    g.map_dataframe(sns.kdeplot, x='error', common_norm=False)
    g.add_legend()
    g.set_titles('{col_name}')
    g.fig.subplots_adjust(top=0.8)
    g.fig.suptitle(f'Residual KDE By Version And Model Family: {target}')
    g.savefig(FIGURES_DIR / f'residual_kde_{target}.png', dpi=180)
    plt.close(g.fig)

## Prediction Error Improvement Relative To Baseline

This view shows how much error reduction each version provides relative to the strongest baseline for the same target/model family.

In [22]:
baseline_mae_lookup = stage_summary[stage_summary['version_stage'] == 'baseline_ref'][['target', 'model_family', 'mae']].rename(columns={'mae': 'baseline_ref_mae'})
improvement_df = stage_summary.merge(baseline_mae_lookup, on=['target', 'model_family'], how='left')
improvement_df['mae_gain_vs_baseline'] = improvement_df['baseline_ref_mae'] - improvement_df['mae']
improvement_df['mae_pct_gain_vs_baseline'] = improvement_df['mae_gain_vs_baseline'] / improvement_df['baseline_ref_mae']
improvement_df.head(20)

,target,model_family,version_stage,n,mae,rmse,median_abs_error,p90_abs_error,baseline_ref_mae,mae_gain_vs_baseline,mae_pct_gain_vs_baseline
0,target_ast,LinearRegression,baseline_ref,26306,1.372025,1.863006,1.036159,2.937483,1.372025,0.000000,0.000000
1,target_ast,LinearRegression,V0,26306,1.349590,1.838608,1.011906,2.876542,1.372025,0.022435,0.016351
2,target_ast,LinearRegression,V1,26306,1.352002,1.830642,1.033497,2.864879,1.372025,0.020022,0.014593
3,target_ast,LinearRegression,V2,26306,1.348317,1.826779,1.026891,2.882087,1.372025,0.023708,0.017279
4,target_ast,RandomForestRegressor,baseline_ref,26306,1.409654,1.910595,1.072534,3.058304,1.409654,0.000000,0.000000
5,target_ast,RandomForestRegressor,V0,26306,1.390801,1.879293,1.056830,2.960596,1.409654,0.018854,0.013375
6,target_ast,RandomForestRegressor,V1,26306,1.384068,1.862216,1.056539,2.952112,1.409654,0.025587,0.018151
7,target_ast,RandomForestRegressor,V2,26306,1.378384,1.849359,1.051774,2.933125,1.409654,0.031270,0.022183
8,target_ast,XGBRegressor,baseline_ref,26306,1.378558,1.868841,1.029394,2.961119,1.378558,0.000000,0.000000
9,target_ast,XGBRegressor,V0,26306,1.364798,1.861474,1.015166,2.954318,1.378558,0.013760,0.009981


In [23]:
for target in sorted(improvement_df['target'].unique()):
    plot_df = improvement_df[(improvement_df['target'] == target) & (improvement_df['version_stage'] != 'baseline_ref')].copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=plot_df, x='version_stage', y='mae_gain_vs_baseline', hue='model_family', ax=ax)
    ax.set_title(f'MAE Gain Vs Baseline: {target}')
    ax.axhline(0, color='black', linewidth=1)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'mae_gain_vs_baseline_{target}.png', dpi=180)
    plt.close(fig)

## Focused View: Monotonic Improvement Cases Only

In [24]:
monotonic_eval = analysis_eval.merge(monotonic_df[['target', 'model_family']], on=['target', 'model_family'], how='inner')
monotonic_stage_summary = stage_summary.merge(monotonic_df[['target', 'model_family']], on=['target', 'model_family'], how='inner')
monotonic_stage_summary[['target', 'model_family', 'version_stage', 'mae', 'rmse']].sort_values(['target', 'model_family', 'version_stage'])

,target,model_family,version_stage,mae,rmse
0,target_ast,RandomForestRegressor,baseline_ref,1.409654,1.910595
1,target_ast,RandomForestRegressor,V0,1.390801,1.879293
2,target_ast,RandomForestRegressor,V1,1.384068,1.862216
3,target_ast,RandomForestRegressor,V2,1.378384,1.849359
4,target_ast,XGBRegressor,baseline_ref,1.378558,1.868841
5,target_ast,XGBRegressor,V0,1.364798,1.861474
6,target_ast,XGBRegressor,V1,1.357088,1.839999
7,target_ast,XGBRegressor,V2,1.353155,1.833953
8,target_pts,LinearRegression,baseline_ref,4.748928,6.195027
9,target_pts,LinearRegression,V0,4.654190,6.083767


In [25]:
for target in sorted(monotonic_stage_summary['target'].unique()):
    plot_df = monotonic_stage_summary[monotonic_stage_summary['target'] == target].copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(data=plot_df, x='version_stage', y='mae', hue='model_family', marker='o', ax=ax)
    ax.set_title(f'Monotonic Improvement Cases: {target}')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'monotonic_mae_progression_{target}.png', dpi=180)
    plt.close(fig)

## Where Predictions Work Well Versus Poorly

In [26]:
best_eval_predictions['error_bucket'] = pd.qcut(best_eval_predictions['abs_error'], q=4, labels=['best_quartile', 'good_quartile', 'bad_quartile', 'worst_quartile'])
bucket_summary = best_eval_predictions.groupby(['target', 'error_bucket']).agg(
    n=('game_id', 'size'),
    mean_actual=('actual', 'mean'),
    mean_prediction=('prediction', 'mean'),
    mean_abs_error=('abs_error', 'mean')
).reset_index()
bucket_summary

,target,error_bucket,n,mean_actual,mean_prediction,mean_abs_error
0,target_ast,best_quartile,10101,1.851005,1.929868,0.369305
1,target_ast,good_quartile,8708,1.984497,2.329632,1.176088
2,target_ast,bad_quartile,5788,3.085867,3.265836,2.326814
3,target_ast,worst_quartile,1709,6.713283,4.583643,4.698366
4,target_pts,best_quartile,2830,9.450177,9.479186,0.373923
5,target_pts,good_quartile,3419,9.168178,9.355160,1.235050
6,target_pts,bad_quartile,6117,8.547164,9.228622,2.521756
7,target_pts,worst_quartile,13940,12.169082,12.379067,7.215850
8,target_reb,best_quartile,6799,3.507428,3.553654,0.380287
9,target_reb,good_quartile,7602,3.255064,3.544660,1.213747


## Worst Misses

In [27]:
worst_misses = best_eval_predictions.sort_values('abs_error', ascending=False)[['target', 'model_version', 'model_family', 'player_name', 'game_date', 'team_abbr', 'opponent_team', 'actual', 'prediction', 'error', 'abs_error']].head(40)
worst_misses

,target,model_version,model_family,player_name,game_date,team_abbr,opponent_team,actual,prediction,error,abs_error
5034,target_pts,V2,XGBRegressor,Pat Connaughton,2025-04-13,MIL,DET,43,6.960547,-36.039453,36.039453
5665,target_pts,V2,XGBRegressor,Jamal Murray,2025-02-12,DEN,POR,55,19.298389,-35.701611,35.701611
3968,target_pts,V2,XGBRegressor,Nikola Jokić,2025-04-01,DEN,MIN,61,28.118279,-32.881721,32.881721
6625,target_pts,V2,XGBRegressor,De'Aaron Fox,2024-11-15,SAC,MIN,60,27.353216,-32.646784,32.646784
14110,target_pts,V2,XGBRegressor,Payton Pritchard,2025-03-05,BOS,POR,43,11.567879,-31.432121,31.432121
894,target_pts,V2,XGBRegressor,Stephen Curry,2025-02-27,GSW,ORL,56,24.743704,-31.256296,31.256296
907,target_pts,V2,XGBRegressor,Stephen Curry,2025-04-01,GSW,MEM,52,21.117884,-30.882116,30.882116
11993,target_pts,V2,XGBRegressor,Quentin Grimes,2025-03-01,PHI,GSW,44,13.663198,-30.336802,30.336802
17455,target_pts,V2,XGBRegressor,Aaron Wiggins,2025-02-01,OKC,SAC,41,10.705419,-30.294581,30.294581
12704,target_pts,V2,XGBRegressor,Anthony Edwards,2025-01-04,MIN,DET,53,23.938669,-29.061331,29.061331


## Save Analysis Outputs

In [28]:
progression_df.to_csv(RESULTS_DIR / 'prediction_progression_summary.csv', index=False)
monotonic_df.to_csv(RESULTS_DIR / 'prediction_monotonic_improvement_cases.csv', index=False)
stage_summary.to_csv(RESULTS_DIR / 'prediction_stage_summary.csv', index=False)
improvement_df.to_csv(RESULTS_DIR / 'prediction_improvement_vs_baseline.csv', index=False)
bucket_summary.to_csv(RESULTS_DIR / 'prediction_error_buckets.csv', index=False)
worst_misses.to_csv(RESULTS_DIR / 'prediction_worst_misses.csv', index=False)
analysis_eval.to_parquet(RESULTS_DIR / 'analysis_eval_predictions.parquet', index=False)
sorted(p.name for p in RESULTS_DIR.glob('prediction_*'))

['prediction_error_buckets.csv',
 'prediction_improvement_vs_baseline.csv',
 'prediction_monotonic_improvement_cases.csv',
 'prediction_progression_summary.csv',
 'prediction_stage_summary.csv',
 'prediction_worst_misses.csv']